In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import resample
from sklearn.metrics import accuracy_score, classification_report

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Add, Concatenate
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report, f1_score


In [ ]:
# During the Adaptation phase, all network weights are frozen except the last layer
# (the one with softmax), whose weights are updated.
# Consequence: limited adaptation.


# ======= RESNET DEFINITION ======= #

def create_resnet(input_dim,
    num_classes,
    n_blocks=5,
    dropout_pcg=0.25,
    factors=256
):
    inp = Input(shape=(input_dim,))

    out = Dense(factors, activation="relu")(inp)
    out = BatchNormalization()(out)
    out = Dropout(dropout_pcg)(out)

    levels = [out]

    # K residual blocks
    for _ in range(n_blocks):
        res = Dense(factors, activation="relu")(out)
        res = BatchNormalization()(res)
        res = Dropout(dropout_pcg)(res)

        out = Add()([out, res])

        out = Dense(factors, activation="relu")(out)
        out = BatchNormalization()(out)
        out = Dropout(dropout_pcg)(out)

        levels.append(out)

    # Concatenation
    out = Concatenate()(levels + [inp])

    out = Dense(factors, activation="relu")(out)
    out = BatchNormalization()(out)
    out = Dropout(dropout_pcg)(out)

    # Classificator
    out = Dense(num_classes, activation="softmax")(out)

    m = Model(inputs=inp, outputs=out)

    m.compile(
        loss="categorical_crossentropy",
        optimizer=tf.keras.optimizers.RMSprop(1e-3),
        metrics=["accuracy"]
    )

    return m


# ======= SEED FOR REPRODUCIBILITY ======= #

np.random.seed(42)
tf.random.set_seed(42)


# ======= OUTPUT DIRECTORIES ======= #
results_dir = "results"
os.makedirs(results_dir, exist_ok=True)

best_resnet_dir = "best_model_resnet"
os.makedirs(best_resnet_dir, exist_ok=True)

merged_dir = "merged_PSD_dataset"
os.makedirs(merged_dir, exist_ok=True)

log_path = os.path.join(
    results_dir,
    "RESNET_cross_subject_adaptation_last_dense_log.txt"
)



def print_and_save(text, f):
    print(text)
    f.write(text + "\n")


# ======= TRAINING DATASET (15 SUBJECT) ======= #

train_files = [
    r'../PSD_data/PSD_EEG_Subject_1.csv',
    r'../PSD_data/PSD_EEG_Subject_2.csv',
    r'../PSD_data/PSD_EEG_Subject_3.csv',
    r'../PSD_data/PSD_EEG_Subject_4.csv',
    r'../PSD_data/PSD_EEG_Subject_5.csv',
    r'../PSD_data/PSD_EEG_Subject_6.csv',
    r'../PSD_data/PSD_EEG_Subject_7.csv',
    r'../PSD_data/PSD_EEG_Subject_8.csv',
    r'../PSD_data/PSD_EEG_Subject_9.csv',
    r'../PSD_data/PSD_EEG_Subject_10.csv',
    r'../PSD_data/PSD_EEG_Subject_11.csv',
    r'../PSD_data/PSD_EEG_Subject_12.csv',
    r'../PSD_data/PSD_EEG_Subject_13.csv',
    r'../PSD_data/PSD_EEG_Subject_14.csv',
    r'../PSD_data/PSD_EEG_Subject_15.csv'
]

# Merge the 15 subjects datasets
df_list = [pd.read_csv(file) for file in train_files]
psd_15 = pd.concat(df_list, ignore_index=True)
psd_15.to_csv(os.path.join(merged_dir, "PSD_dataset_15.csv"), index=False)


# ======= TRAINING SET LOADING ======= #

df = psd_15

# PSD features (all columns except Event Id)
X = df.drop("Event Id", axis=1).astype(float).values
# Labels
y = df["Event Id"].values

input_dim = X.shape[1]

# Class mapping (derived ONLY from training data)
classes = np.array(sorted(np.unique(y)))
num_classes = len(classes)
to_index = {c: i for i, c in enumerate(classes)}


# ======= TRAIN/VALIDATION SPLIT ======= #

# Only for training
Xtr, Xval, ytr, yval = train_test_split(
    X, y,
    train_size=0.8,
    shuffle=True,
    stratify=y,
    random_state=42
)


# ======= SCALER ======= #

# Fit ONLY on training set (15 subjects)
scaler = MinMaxScaler().fit(Xtr)
Xtr_s  = scaler.transform(Xtr)
Xval_s = scaler.transform(Xval)


# ======= OVERSAMPLING ======= #

# ONLY on training set to avoid Data Leakage
df_tr = pd.DataFrame(Xtr_s)
df_tr["Event Id"] = ytr

max_n = df_tr["Event Id"].value_counts().max()
dfs = []
for cls, sub in df_tr.groupby("Event Id"):
    if len(sub) < max_n:
        sub = resample(sub, replace=True, n_samples=max_n, random_state=42)
    dfs.append(sub)

df_bal = pd.concat(dfs).sample(frac=1, random_state=42).reset_index(drop=True)
Xtr_bal = df_bal.drop("Event Id", axis=1).values
ytr_bal = df_bal["Event Id"].values


# ======= ONE-HOT ======= #

ytr_cat  = to_categorical([to_index[c] for c in ytr_bal], num_classes=num_classes)
yval_cat = to_categorical([to_index[c] for c in yval], num_classes=num_classes)


# ======= TRAINING ======= #

model = create_resnet(input_dim, num_classes)

ckpt_path_15 = os.path.join(best_resnet_dir, "best_resnet_15.weights.h5")
ckpt_15 = ModelCheckpoint(
    ckpt_path_15,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=True,
    mode="min",
    verbose=0
)
rlr_15 = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=0
)

model.fit(
    Xtr_bal, ytr_cat,
    validation_data=(Xval_s, yval_cat),
    epochs=150,
    batch_size=64,
    callbacks=[ckpt_15, rlr_15],
    verbose=0
)

# Load the best weights from training on the 15 subjects
model.load_weights(ckpt_path_15)


# ======= TEST DATASETS (12 NEW SUBJECTS) ======= #

test_files = [
    r'../PSD_data/PSD_EEG_Subject_16.csv',
    r'../PSD_data/PSD_EEG_Subject_17.csv',
    r'../PSD_data/PSD_EEG_Subject_18.csv',
    r'../PSD_data/PSD_EEG_Subject_19.csv',
    r'../PSD_data/PSD_EEG_Subject_20.csv',
    r'../PSD_data/PSD_EEG_Subject_21.csv',
    r'../PSD_data/PSD_EEG_Subject_22.csv',
    r'../PSD_data/PSD_EEG_Subject_23.csv',
    r'../PSD_data/PSD_EEG_Subject_24.csv',
    r'../PSD_data/PSD_EEG_Subject_25.csv',
    r'../PSD_data/PSD_EEG_Subject_26.csv',
    r'../PSD_data/PSD_EEG_Subject_27.csv'
]


with open(log_path, "w", encoding="utf-8") as f:

    print_and_save("=" * 70, f)
    print_and_save("CROSS-SUBJECT ADAPTATION TEST", f)
    print_and_save("Train: Subjects 1–15 | Test: Subjects 16–27", f)
    print_and_save("Adaptation: 20% of each new subject", f)
    print_and_save("=" * 70 + "\n", f)

    acc_list = []
    f1_list = []

    for test_file in test_files:

        subject_name = os.path.splitext(os.path.basename(test_file))[0]
        print_and_save("=" * 70, f)
        print_and_save(f"SUBJECT: {subject_name}", f)

        # Reset to base weights (15-subject training)
        model.load_weights(ckpt_path_15)

        # Load new subject dataset
        df_new_sub = pd.read_csv(test_file)

        X_new_sub = df_new_sub.drop("Event Id", axis=1).astype(float).values
        y_new_sub = df_new_sub["Event Id"].values

        # Same normalization used for training (15 subjects)
        X_new_sub_s = scaler.transform(X_new_sub)

        # Label mapping using the 15-subject classes
        mask_known = np.isin(y_new_sub, classes)
        X_new_sub_s = X_new_sub_s[mask_known]
        y_new_sub = y_new_sub[mask_known]
        y_new_sub_idx = np.array([to_index[c] for c in y_new_sub])


        # ======= SPLIT ======= #

        # 20% Adaptation, 80% Test
        X_ft, X_test, y_ft_idx, y_test_idx = train_test_split(
            X_new_sub_s, y_new_sub_idx,
            train_size=0.2,
            shuffle=True,
            stratify=y_new_sub_idx,
            random_state=42
        )

        # One-hot ONLY for Adaptation
        y_ft_cat = to_categorical(y_ft_idx, num_classes=num_classes)


        # ======= ADAPTATION ======= #

        for layer in model.layers:
            layer.trainable = False
        model.layers[-1].trainable = True

        model.compile(
            loss="categorical_crossentropy",
            optimizer=tf.keras.optimizers.RMSprop(1e-4),
            metrics=["accuracy"]
        )

        ckpt_ft_path = os.path.join(
            best_resnet_dir,
            f"best_resnet_Adaptation_{subject_name}.weights.h5"
        )
        ckpt_ft = ModelCheckpoint(
            ckpt_ft_path,
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=True,
            mode="min",
            verbose=0
        )
        rlr_ft = ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-6,
            verbose=0
        )

        model.fit(
            X_ft, y_ft_cat,
            validation_split=0.2,
            epochs=60,
            batch_size=64,
            callbacks=[ckpt_ft, rlr_ft],
            verbose=0
        )

        model.load_weights(ckpt_ft_path)


        # ======= TEST ======= #

        ypred = model.predict(X_test, verbose=0)
        ypred_lab = np.argmax(ypred, axis=1)

        acc = accuracy_score(y_test_idx, ypred_lab)
        f1m = f1_score(y_test_idx, ypred_lab, average="macro")

        acc_list.append(acc)
        f1_list.append(f1m)

        print_and_save(f"Test Accuracy AFTER Adaptation: {acc:.4f}", f)
        print_and_save(f"Macro-F1 AFTER Adaptation: {f1m:.4f}", f)

        print_and_save(
            classification_report(
                y_test_idx,
                ypred_lab,
                labels=np.arange(num_classes),
                target_names=[str(c) for c in classes],
                zero_division=0
            ),
            f
        )
        print_and_save("", f)

    print_and_save("=" * 70, f)
    print_and_save(f"MEAN ACCURACY (12 subjects): {np.mean(acc_list):.4f}", f)
    print_and_save(f"MEAN MACRO-F1 (12 subjects): {np.mean(f1_list):.4f}", f)
    print_and_save("=" * 70, f)